In [ ]:
import time
import pandas as pd
import fasttext
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # resultado fijo

In [ ]:
modelo_ft = fasttext.load_model("../../models/lid.176.ftz")

In [ ]:
# Detectores 
def detectar_fasttext(texto):
    texto = texto.replace("\n", " ") + "\n"  # fastText necesita un solo salto de línea, al final
    resultado = modelo_ft.f.predict(texto, 1, 0.0, "strict")
    return resultado[0][1].replace("__label__", "")


def detectar_langdetect(texto):
    try:
        return detect(texto)
    except Exception:  # textos sin letras
        return "desconocido"

In [4]:
# Prueba
print(detectar_fasttext("hola como estas"), detectar_langdetect("hola como estas"))
print(detectar_fasttext("A comida estava ótima"), detectar_langdetect("A comida estava ótima"))
print(detectar_fasttext("The food was great"), detectar_langdetect("The food was great"))

es es
pt pt
en en


In [5]:
# Carga de datos
en = pd.read_csv("../../data/separacion_reseñas/resenas_ingles.csv")
es = pd.read_csv("../../data/traducidas/resenas_espanol_trad.csv")
pt = pd.read_csv("../../data/traducidas/resenas_portugues_trad.csv")

In [6]:
# Unir idiomas
datos = pd.concat([
    pd.DataFrame({"texto": en["Review Text"], "idioma": "en"}),
    pd.DataFrame({"texto": es["Review Text_es"], "idioma": "es"}),
    pd.DataFrame({"texto": pt["Review Text_pt-BR"], "idioma": "pt"}),
], ignore_index=True)

datos["texto"] = datos["texto"].astype(str)
print(len(datos), "reseñas")

19896 reseñas


In [7]:
# fastText
inicio = time.time()
datos["fasttext"] = datos["texto"].apply(detectar_fasttext)
tiempo_ft = time.time() - inicio

In [8]:
# langdetect
inicio = time.time()
datos["langdetect"] = datos["texto"].apply(detectar_langdetect)
tiempo_ld = time.time() - inicio

In [9]:
# Aciertos y velocidad
aciertos_ft = (datos["fasttext"] == datos["idioma"]).mean() * 100
aciertos_ld = (datos["langdetect"] == datos["idioma"]).mean() * 100

print(f"fastText:   {aciertos_ft:.2f}% de aciertos, {tiempo_ft / len(datos) * 1000:.2f} ms por reseña")
print(f"langdetect: {aciertos_ld:.2f}% de aciertos, {tiempo_ld / len(datos) * 1000:.2f} ms por reseña")

fastText:   99.98% de aciertos, 0.16 ms por reseña
langdetect: 99.93% de aciertos, 6.07 ms por reseña


In [10]:
# Aciertos por idioma
datos["acierto_fasttext"] = datos["fasttext"] == datos["idioma"]
datos["acierto_langdetect"] = datos["langdetect"] == datos["idioma"]

datos.groupby("idioma")[["acierto_fasttext", "acierto_langdetect"]].mean() * 100

,acierto_fasttext,acierto_langdetect
idioma,,
en,100.000000,99.954765
es,100.000000,99.969843
pt,99.954765,99.879373


In [11]:
# Reseñas cortas
muestra = datos.sample(n=3000, random_state=42)

for palabras in [3, 5, 10]:
    cortas = muestra["texto"].str.split().str[:palabras].str.join(" ")
    ft = (cortas.apply(detectar_fasttext) == muestra["idioma"]).mean() * 100
    ld = (cortas.apply(detectar_langdetect) == muestra["idioma"]).mean() * 100
    print(f"{palabras} palabras -> fastText: {ft:.1f}%   langdetect: {ld:.1f}%")

3 palabras -> fastText: 87.5%   langdetect: 66.7%


5 palabras -> fastText: 94.4%   langdetect: 84.6%


10 palabras -> fastText: 98.7%   langdetect: 97.1%
